# 02 - Crop Group Trends (Gold)

One row per year, quarter, and crop group (PALAY, CORN, OTHER).
Year-over-year growth uses the same quarter one year earlier.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from configs.project_config import SILVER_TABLES, GOLD_TABLES

source = SILVER_TABLES['crop_production_enriched']
target = GOLD_TABLES['crop_group_trends']

df = spark.table(source)

trends = (
    df.groupBy('year', 'quarter', 'quarter_num', 'crop_group')
    .agg(
        F.round(F.sum('production'), 3).alias('total_production'),
        F.countDistinct('canonical_province').alias('province_count')
    )
)

w = Window.partitionBy('crop_group', 'quarter').orderBy('year')
trends = trends.withColumn(
    'yoy_growth',
    F.when(
        (F.lag('total_production', 1).over(w).isNotNull())
        & (F.lag('total_production', 1).over(w) != 0),
        F.round(
            (F.col('total_production') - F.lag('total_production', 1).over(w))
            / F.lag('total_production', 1).over(w),
            4
        )
    )
).drop('quarter_num')

(
    trends.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(target)
)

out = spark.table(target)
print(f'Wrote: {target}')
print(f'Rows : {out.count()}')
out.groupBy('crop_group').count().orderBy('crop_group').show()